In [ ]:
%%capture
# uv sync --group develop --group lint --group test --group notebook
import os
from pathlib import Path

import pandas as pd
from dj_notebook import activate
from django_pandas.io import read_frame

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from edc_visit_schedule.models import SubjectScheduleHistory

from meta_prn.models import OffSchedule, OffStudyMedication, PregnancyNotification

In [ ]:
df = read_frame(PregnancyNotification.objects.all())
df_off = read_frame(OffSchedule.objects.all())

In [ ]:
df_schedule = read_frame(SubjectScheduleHistory.objects.all())

In [ ]:
df = df[["subject_identifier", "report_datetime"]].merge(df_schedule.query("schedule_name=='schedule_pregnancy'")[["subject_identifier", "visit_schedule_name", "schedule_name", "onschedule_datetime", "offschedule_datetime"]], on="subject_identifier", how="left")


In [ ]:
df_withdrawal = read_frame(OffStudyMedication.objects.all())

In [ ]:
df_withdrawal["date_diff"] = df_withdrawal["stop_date"] - df_withdrawal["last_dose_date"]
df_withdrawal = df_withdrawal[["subject_identifier", "stop_date", "last_dose_date", "date_diff", "reason", "reason_other"]]
df_withdrawal.loc[df_withdrawal["reason"]=="Other reason (specify below)", "reason"] = df_withdrawal["reason_other"]
df_withdrawal = df_withdrawal.drop(columns=["reason_other"])

In [ ]:
df = df.merge(df_withdrawal[["subject_identifier", "stop_date", "last_dose_date", "date_diff", "reason"]], on=["subject_identifier"], how="left")

In [ ]:
df